In [1]:
import importlib
import torch

import utils.final_run_executor_tmp as final_run_executor_tmp
import data_provider.data_factory_tmp as data_factory_tmp
import data_provider.data_loader_tmp as data_loader_tmp
import engine.evaluator_tmp as evaluator_tmp
import utils.visualization_tmp as visualization_tmp
import utils.experiment_helpers_tmp as experiment_helpers_tmp

importlib.reload(data_loader_tmp)
importlib.reload(data_factory_tmp)
importlib.reload(evaluator_tmp)
importlib.reload(visualization_tmp)
importlib.reload(final_run_executor_tmp)
importlib.reload(experiment_helpers_tmp)

from utils.experiment_helpers_tmp import (
    build_base_config,
    make_experiment_grid,
    run_train_and_eval_loop,
    run_eval_only_loop,
)

print("Reload completed.")
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

Reload completed.
Device: cuda


## Optional: evaluate only already-trained runs in memory

Use this cell only if `train_results` already exists in the current kernel.
It will add `test_outputs.npz` and plots without retraining.

In [ ]:
# OPTIONAL: eval only on already existing train_results
# Requires that train_results already exists in notebook memory.

# eval_plot_bundle = run_eval_only_loop(
#     train_results=train_results,
#     save_npz_splits=("test",),
#     make_test_plot=True,
#     plot_num_random=3,
#     plot_seed=40,
#     plot_channel_idx=-1,
#     include_val=False,
#     verbose=False,
# )

## Build the full 24-config grid

24 = 2 modes × 2 datasets × 3 models × 2 token styles

In [2]:
ROOT_DIR_RUNS = "runs_debug"

experiment_grid_24 = make_experiment_grid(
    experiment_modes=["multivariate", "increasing_lookback"],
    datasets=["ETTh1", "ECL"],
    models=["gru", "transformer", "mamba"],
    token_styles=["time", "variate"],
)

print("Num configs:", len(experiment_grid_24))
print("First 5:", experiment_grid_24[:5])

Num configs: 24
First 5: [('multivariate', 'ETTh1', 'gru', 'time'), ('multivariate', 'ETTh1', 'gru', 'variate'), ('multivariate', 'ETTh1', 'transformer', 'time'), ('multivariate', 'ETTh1', 'transformer', 'variate'), ('multivariate', 'ETTh1', 'mamba', 'time')]


In [6]:
experiment_grid_24[-5:]

[('increasing_lookback', 'ECL', 'gru', 'variate'),
 ('increasing_lookback', 'ECL', 'transformer', 'time'),
 ('increasing_lookback', 'ECL', 'transformer', 'variate'),
 ('increasing_lookback', 'ECL', 'mamba', 'time'),
 ('increasing_lookback', 'ECL', 'mamba', 'variate')]

## Seed 42: full 24 configs with NPZ + plots

In [ ]:
# # already done in run.ipynb !!!

# base_cfg_seed42 = build_base_config(
#     seed=42,
#     batch_size=32,
#     stride=1,
#     ratios=(0.7, 0.15, 0.15),
#     min_lr=1e-7,
#     early_stop_patience=3,
#     num_epochs=10,
#     verbose=False,
# )

# bundle_seed42 = run_train_and_eval_loop(
#     experiment_grid=experiment_grid_24,
#     base_cfg=base_cfg_seed42,
#     root_dir_runs=ROOT_DIR_RUNS,
#     eval_save_npz_splits=("test",),
#     eval_make_test_plot=True,
#     eval_plot_num_random=3,
#     eval_plot_seed=40,
#     eval_plot_channel_idx=-1,
#     eval_include_val=False,
#     verbose_train=False,
#     verbose_eval=False,
# )

## Remaining 2 seeds: JSON-only evaluation

Change the seed list if needed.
This runs the same 24 configs for each additional seed, but saves only `post_training_metrics.json`.

In [3]:

for seed in [7, 123]:
    print(f"\n########## SEED = {seed} ##########")

    base_cfg = build_base_config(
        seed=seed,
        batch_size=32,
        stride=1,
        ratios=(0.7, 0.15, 0.15),
        min_lr=1e-7,
        early_stop_patience=3,
        num_epochs=10,
        verbose=False,
    )

    bundle = run_train_and_eval_loop(
        experiment_grid=experiment_grid_24,
        base_cfg=base_cfg,
        root_dir_runs=ROOT_DIR_RUNS,
        eval_save_npz_splits=None,   # JSON only
        eval_make_test_plot=False,
        eval_plot_num_random=3,
        eval_plot_seed=40,
        eval_plot_channel_idx=-1,
        eval_include_val=False,
        verbose_train=False,
        verbose_eval=False,
    )

    print("Seed completed.")


########## SEED = 7 ##########

[1/24] mode=multivariate | ETTh1 | gru | time | L96_T96 | BS=32 | seed=7
TRAIN OK  | best_val=0.487219 | best_epoch=1
          saved: runs_debug/ETTh1/time/gru/L96_T96/seed_7/2026-03-15_00-04
EVAL  OK  | test mse=0.724700
          test dir: runs_debug/ETTh1/time/gru/L96_T96/seed_7/2026-03-15_00-04/test

[2/24] mode=multivariate | ETTh1 | gru | variate | L96_T96 | BS=32 | seed=7
TRAIN OK  | best_val=0.378283 | best_epoch=0
          saved: runs_debug/ETTh1/variate/gru/L96_T96/seed_7/2026-03-15_00-04
EVAL  OK  | test mse=0.437999
          test dir: runs_debug/ETTh1/variate/gru/L96_T96/seed_7/2026-03-15_00-04/test

[3/24] mode=multivariate | ETTh1 | transformer | time | L96_T96 | BS=32 | seed=7
TRAIN OK  | best_val=0.442365 | best_epoch=2
          saved: runs_debug/ETTh1/time/transformer/L96_T96/seed_7/2026-03-15_00-04
EVAL  OK  | test mse=0.560569
          test dir: runs_debug/ETTh1/time/transformer/L96_T96/seed_7/2026-03-15_00-04/test

[4/24] mode=m

## aggregate results

In [8]:
import importlib
import utils.aggregate_results_tmp as aggregate_results_tmp

importlib.reload(aggregate_results_tmp)
from utils.aggregate_results_tmp import build_aggregate_artifacts

agg_info = build_aggregate_artifacts(root_dir_runs="runs_debug")
print(agg_info)


{'paths': {'all_runs_by_seed_json': 'runs_debug/_aggregates/all_runs_by_seed.json', 'all_runs_by_seed_csv': 'runs_debug/_aggregates/all_runs_by_seed.csv', 'all_configs_mean_std_json': 'runs_debug/_aggregates/all_configs_mean_std.json', 'all_configs_mean_std_csv': 'runs_debug/_aggregates/all_configs_mean_std.csv'}, 'num_seed_records': 72, 'num_config_records': 24}


### Load the two CSVs

In [9]:
import pandas as pd

df_runs = pd.read_csv("runs_debug/_aggregates/all_runs_by_seed.csv")
df_cfgs = pd.read_csv("runs_debug/_aggregates/all_configs_mean_std.csv")

print(df_runs.shape)
print(df_cfgs.shape)


(72, 22)
(24, 31)


In [16]:
df_cfgs.sort_values("test_mse_mean").head(10)


,dataset,mode,model,token_style,window,horizon,seeds,train_mse_mean,train_mse_std,train_mae_mean,...,epochs_ran_mean,epochs_ran_std,training_time_sec_mean,training_time_sec_std,ms_per_iter_mean,ms_per_iter_std,gpu_mem_gb_mean,gpu_mem_gb_std,num_trainable_params,size_mb
5,ECL,increasing_lookback,transformer,variate,192,96,"7,42,123",0.114648,0.002192,0.222304,...,10.000000,0.000000,244.838159,9.366049,7.065221,1.522405,0.128106,0.000930,6456416,24.629272
3,ECL,increasing_lookback,mamba,variate,192,96,"7,42,123",0.134602,0.000162,0.243816,...,10.000000,0.000000,73.595379,6.745519,6.281590,0.784880,0.045735,0.000210,1186912,4.527710
1,ECL,increasing_lookback,gru,variate,192,96,"7,42,123",0.138205,0.000199,0.248609,...,10.000000,0.000000,414.685605,11.390362,41.473432,3.985077,0.297819,0.000311,17998944,68.660522
11,ECL,multivariate,transformer,variate,96,96,"7,42,123",0.127623,0.001801,0.235478,...,10.000000,0.000000,237.966149,8.375222,6.972377,1.528629,0.124026,0.000448,6407264,24.441772
7,ECL,multivariate,gru,variate,96,96,"7,42,123",0.133952,0.002180,0.247939,...,10.000000,0.000000,410.829945,6.693053,41.461803,3.977688,0.295841,0.000825,17949792,68.473022
9,ECL,multivariate,mamba,variate,96,96,"7,42,123",0.142611,0.001338,0.251830,...,10.000000,0.000000,65.800288,5.987883,6.082127,0.866185,0.043746,0.000353,1162336,4.433960
0,ECL,increasing_lookback,gru,time,192,96,"7,42,123",0.100108,0.004314,0.219928,...,10.000000,0.000000,160.639195,6.272901,5.488302,1.117832,0.598739,0.000412,38213728,145.773804
6,ECL,multivariate,gru,time,96,96,"7,42,123",0.092036,0.003500,0.211399,...,10.000000,0.000000,114.117974,5.187722,4.158896,0.795194,0.595775,0.001469,38213728,145.773804
4,ECL,increasing_lookback,transformer,time,192,96,"7,42,123",0.104506,0.003168,0.227880,...,9.333333,0.577350,330.806538,19.464941,5.294664,1.227773,0.903338,0.000117,57108576,217.851929
8,ECL,multivariate,mamba,time,96,96,"7,42,123",0.110597,0.030161,0.229529,...,7.000000,2.645751,97.444832,47.272839,14.999752,1.815080,0.320382,0.001125,19579104,74.688354


In [17]:
df_cfgs.groupby("token_style")["test_mse_mean"].mean()


token_style
time       0.457569
variate    0.295270
Name: test_mse_mean, dtype: float64

In [18]:
df_cfgs.groupby("model")["test_mse_mean"].mean()


model
gru            0.381545
mamba          0.387180
transformer    0.360533
Name: test_mse_mean, dtype: float64

In [19]:
import seaborn as sns
sns.barplot(data=df_cfgs, x="model", y="test_mse_mean", hue="mode")


<Axes: xlabel='model', ylabel='test_mse_mean'>